# Titanic · baseline com validação estratificada

Notebook executável na plataforma Kaggle após adicionar os dados oficiais da competição. **Não significa submissão já realizada.**

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
train[['Survived', 'Pclass', 'Sex', 'Age']].head()

## Pipeline sem vazamento

Imputação, one-hot encoding e classificador são ajustados separadamente durante cada fold.

In [ ]:
numeric = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical = ['Sex', 'Embarked']
features = numeric + categorical
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical),
])
model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, train[features], train['Survived'], scoring='accuracy', cv=cv)
print(f'Accuracy CV: {scores.mean():.4f} ± {scores.std():.4f}')

## Treino final e arquivo de submissão

O conjunto de teste do Kaggle não é usado na validação local.

In [ ]:
model.fit(train[features], train['Survived'])
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': model.predict(test[features]).astype(int)})
from pathlib import Path
submission.to_csv('/kaggle/working/submission.csv', index=False)
assert Path('/kaggle/working/submission.csv').is_file()
print('Ready for Kaggle submission: /kaggle/working/submission.csv')
submission.head()

**Para participar de fato:** execute, gere o CSV e envie na aba *Submit Predictions* da competição. Só então registre o URL público e o score obtido.

## Enviar o arquivo para a competição

Selecione **Save Version → Save & Run All** e aguarde a execução completa. Na versão salva, confira **Output → submission.csv**. Uma versão *Quick Save* não garante que o notebook seja executado nem que os arquivos de saída sejam salvos. Depois selecione `submission.csv` em **Submit Predictions**.

Este notebook gera previsões, mas **não submete automaticamente** ao Kaggle. O score do leaderboard só existe após submissão e avaliação pela plataforma.